<div align="right"><a href="S34-Burg-Algorithm.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L34-Burg-Algorithmus.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L33 Yule-Walker-/Autokorrelationsmethode](./L33-Yule-Walker-und-Autokorrelationsmethode.ipynb) | [Aufgabe 34](./A34-Burg-Algorithmus.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L35 Vergleich von Spektralschätzverfahren](./L35-SpektraleSchaetzverfahren.ipynb)

</div>

---


## Aufgabe: Burg

Die Grundidee des Burg-Algorithmus besteht darin, die Leistung  
des Vorwärts- und Rückwärtsprädiktionsfehlers auszubalancieren  
und beide Leistungen gemeinsam zu minimieren.  
Dieser Ansatz liefert eine unverzerrte Schätzung der Koeffizienten  
des AR($n$)-Modells  
(im Gegensatz zur Autokorrelationsmethode); zusätzlich ist der Analysefilter  
immer ein Minimum-Phase-Filter  
(im Gegensatz zur Autokovarianzmethode).

Wir beginnen mit einem (All-Pole-)System  

$H(z) = \frac{1}{A(z)}$  

mit  

$$
A(z) = 1 + 0.3611z^{-1} - 0.1373z^{-2} + 0.2685z^{-3}
- 0.0533z^{-4} - 0.3353z^{-5} - 0.307z^{-6}
$$

der Ordnung $m = 6$, das durch einen stationären, nullmittelwertigen  
weißen Rauschprozess $N(k)$ mit Leistung $\sigma_N^2 = 1$ angeregt wird  
(z. B. unter Verwendung von `scipy.signal.lfilter`).

<img src="./pngs/l15sys.eps" style="width:70%;">

---

### a)

Stellen Sie die Impulsantwort $h(k)$ des Systems im Intervall  

$$
0 \le k \le 80
$$

dar, 
und 
und erstellen Sie das Pol-Nullstellen-Diagramm mit der bereitgestellten Funktion `zplane`  
(aus `helper_functions.py`).

---

### b)

Schätzen Sie die Koeffizienten eines AR-Systems AR($n$)

$$
\frac{1}{A_n(z)} = \frac{1}{a_0 + a_1 z^{-1} + \cdots + a_n z^{-n}}
$$

auf Basis von $N = 2^{13}$ Samples des Prozesses $X(k)$  
für verschiedene Modellordnungen $n = 2,\,6,\,32$  
(z. B. unter Verwendung einer in Python verfügbaren Burg-Algorithmus-Implementierung).

Vergleichen Sie die geschätzten Leistungsdichtespektren miteinander  
und mit dem exakten Leistungsdichtespektrum  
$S_{XX}(e^{j\Omega})$.

Warum ist die Schätzung insbesondere für $n = 6$ gut?

---

### c)

Wiederholen Sie die Aufgaben a) und b) für ein FIR-System

$$
H(z) = 1 - 4z^{-1} + 6z^{-2} - 4z^{-3} + z^{-4}
$$

der Ordnung $m = 4$.

Beachten Sie die Anordnung der Pole des AR($n$)-Modells,  
die dazu dienen, die vierfache Nullstelle von $H(z)$ bei $z = 1$ auszugleichen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import zplane, burg_algo
# BEGIN SOLUTION
# ##############################################################################
# ##  Subtask a: AR-System, Transient Response                                ##
# ##############################################################################

MA = np.array([1.0])
AR = np.array([1.0000, 0.3611, -0.1373, 0.2685, -0.0533, -0.3353, -0.307])
NFFT = 2**10

# Impulse response (impz)
t = np.arange(80)
impulse = np.zeros(80)
impulse[0] = 1.0
h_arma = signal.lfilter(MA, AR, impulse)

# Graphical output
plt.figure(figsize=(10, 4))
plt.stem(t, h_arma)
plt.grid(True)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title(f'Impulse Response of the AR({len(AR)-1}) Model')

# Pole-Zero plot
zplane(MA, AR, title=f'Poles of the AR({len(AR)-1}) Model')
plt.show()

# ##############################################################################
# ##  Subtask b: AR-System, Power Spectral Density (PSD)                      ##
# ##############################################################################

Omega_norm = np.linspace(0, 1, NFFT, endpoint=False)
# Exact PSD calculation
w, h = signal.freqz(MA, AR, worN=NFFT, whole=True)
Sxx_arma = np.abs(h)**2

n_orders = [2, len(AR)-1, 32]
N = 2^13
N_pre = 80 # Transient samples to discard
noise = np.random.randn(N + N_pre)
x_all = signal.lfilter(MA, AR, noise)
x = x_all[N_pre:]

for n in n_orders:
    ar_est = burg_algo(x, n)
    w_est, h_est = signal.freqz([1], ar_est, worN=NFFT, whole=True)
    Sxx_ar = np.abs(h_est)**2
    
    # Normalization as in the MATLAB code
    Sxx_ar = Sxx_ar / np.sum(Sxx_ar) * np.sum(Sxx_arma)
    
    # Plot PSD
    plt.figure()
    plt.plot(Omega_norm, 10 * np.log10(Sxx_arma / np.max(Sxx_arma)), 'r--', label='Exact PSD')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_ar / np.max(Sxx_ar)), 'b-', label='Estimated PSD')
    plt.ylabel('$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel('$\Omega/2\pi$')
    plt.title(f'PSD of Burg-AR({n}), N={N}')
    plt.grid(True)
    plt.legend()
    plt.show()
    
    # Plot Poles for the estimation
    zplane([1], ar_est, title=f'Poles of Burg-AR({n})')
    plt.show()

# ##############################################################################
# ##  Subtask c: MA-System                                                    ##
# ##############################################################################

MA_c = np.array([1, -4, 6, -4, 1])
AR_c = np.array([1.0])

# Impulse response
h_arma_c = signal.lfilter(MA_c, AR_c, np.concatenate(([1], np.zeros(len(MA_c)-1))))
t_c = np.arange(len(h_arma_c))

plt.figure()
plt.stem(t_c, h_arma_c)
plt.grid(True)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title(f'Impulse Response of the MA({len(MA_c)-1}) Model')

# Pole-Zero plot for MA model
zplane(MA_c, AR_c, title=f'Poles/Zeros of the MA({len(MA_c)-1}) Model')
plt.show()

# Analyze filter order AR(n) for the MA system (Spectral Estimation)
w_c, h_c = signal.freqz(MA_c, AR_c, worN=NFFT, whole=True)
Sxx_arma_c = np.abs(h_c)**2

N_pre_c = 5
noise_c = np.random.randn(N + N_pre_c)
x_all_c = signal.lfilter(MA_c, AR_c, noise_c)
x_c = x_all_c[N_pre_c:]

for n in [2, 6, 32]:
    ar_est_c = burg_algo(x_c, n)
    w_est_c, h_est_c = signal.freqz([1], ar_est_c, worN=NFFT, whole=True)
    Sxx_ar_c = np.abs(h_est_c)**2
    
    Sxx_ar_c = Sxx_ar_c / np.sum(Sxx_ar_c) * np.sum(Sxx_arma_c)
    
    # Avoid log(0) for plotting
    Sxx_ar_c[Sxx_ar_c == 0] = 1e-10
    Sxx_arma_c[Sxx_arma_c == 0] = 1e-10
    
    # Plot PSD comparison
    plt.figure()
    plt.plot(Omega_norm, 10 * np.log10(Sxx_arma_c / np.max(Sxx_arma_c)), 'r--', label='Exact PSD')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_ar_c / np.max(Sxx_ar_c)), 'b-', label='Estimated PSD')
    plt.ylim([-70, 0])
    plt.grid(True)
    plt.ylabel('$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel('$\Omega/2\pi$')
    plt.title(f'PSD of Burg-AR({n}) (MA Source), N={N}')
    plt.legend()
    plt.show()
    
    # Plot Poles of the AR approximation
    zplane([1], ar_est_c, title=f'Poles of Burg-AR({n})')
    plt.show()
# END SOLUTION

## Lösung: Burg

### a)

Die folgende Abbildung zeigt die Impulsantwort $h(k)$ des AR(6)-Systems  
im Intervall $0 \le k \le 80$  
sowie das zugehörige Pol-Nullstellen-Diagramm,  
berechnet mit Standardverfahren der Signalverarbeitung in Python  
(wobei für das Pol-Nullstellen-Diagramm die bereitgestellte Funktion `zplane` verwendet wird).  
Die Pole liegen bei

- $z_{\infty,1} = 0.3444 + i\cdot 0.8315$  
- $z_{\infty,2} = z_{\infty,1}^{*}$  
- $z_{\infty,3} = 0.85$  
- $z_{\infty,4} = -0.91$  
- $z_{\infty,5} = -0.49 + i\cdot 0.4950$  
- $z_{\infty,6} = z_{\infty,5}^{*}$  

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_a1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_a2.png" style="width:45%;">
</div>

---

### b)

Verfahren zur Spektralschätzung (unter Verwendung der Yule–Walker- bzw. Burg-Methode)  
zielen darauf ab, einen AR-gefilterten weißen Rauschprozess zu modellieren,  
d. h. die AR-Parameter  
$\hat{a}_{0,n}, \ldots, \hat{a}_{n,n}$  
zu schätzen, sodass dessen statistische Eigenschaften  
(insbesondere die Autokorrelationsfunktion)  
möglichst gut denen des gegebenen stationären Prozesses $X(k)$  
mit beliebigem Spektrum entsprechen.

Dieser Ansatz funktioniert besonders gut, wenn der zugrunde liegende Prozess  
tatsächlich durch AR-Filterung erzeugt wurde und wenn die Ordnung  
des AR($n$)-Modells der Ordnung des wahren Filters entspricht.  
Der Fall einer Filterung mit einem AR-System der Ordnung $m = 6$  
ist nachfolgend dargestellt.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b12.png" style="width:45%;">
</div>

Die beiden oberen Abbildungen zeigen das exakte Leistungsdichtespektrum  
im Vergleich zu den Leistungsdichtespektren,  
die aus mit dem Burg-Algorithmus geschätzten AR-Koeffizienten  
für die Modellordnungen $n = 2, 6, 32$ resultieren.

Die unteren Abbildungen stellen die zugehörigen Polpositionen dar,  
wobei die Pole des ursprünglichen Systems $H(z)$ explizit  
als *Kreise* markiert sind.  
Alle Berechnungen basieren auf einer einzelnen Realisierung  
der Länge $N = 2^{13} = 8192$,  
was eine statistisch hinreichende Schätzung ermöglicht.

Wird die Ordnung des AR-Modells unterschätzt (z. B. $n = 2$),  
versucht der Burg-Algorithmus, einen möglichst guten Kompromiss  
bei der Platzierung der Pole zu finden.  
Dennoch verbleibt eine deutliche Abweichung  
zwischen dem geschätzten und dem exakten Leistungsdichtespektrum.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b22.png" style="width:45%;">
</div>

Entspricht die Ordnung des AR($n$)-Modells der Ordnung  
des tatsächlichen Filters $H(z)$,  
so ist die Approximationsgüte – wie erwartet – am besten.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b32.png" style="width:45%;">
</div>

Im Fall einer Überabschätzung der Modellordnung mit $n = 32$  
werden alle zusätzlichen Koeffizienten nahezu null geschätzt:

$$
\hat{a}_{n,i} \approx 0 \quad \text{für} \quad i > m
$$

Diese Koeffizienten entsprechen Polen,  
die auf einem Kreis nahe dem Einheitskreis liegen.  
Infolgedessen zeigt das geschätzte Leistungsdichtespektrum  
einen stärker „welligen“ Verlauf.  
Die Verschlechterung der Approximationsgüte bleibt jedoch gering.

---

### c)

Die folgende Abbildung zeigt die Impulsantwort $h(k)$ des MA(4)-Systems  
im Intervall $0 \le k \le 4$.

Da $H(z)$ ein MA-Modell der Ordnung $m = 4$ ist,  
besitzt die Impulsantwort eine endliche Länge von $m + 1 = 5$:

$$
h(k) = \{1, -4, 6, -4, 1\}
$$

Folglich sind die Energiekorrelationsfunktion  
$r_{HH}^{(E)}(\kappa)$  
und damit auch die Autokorrelationsfunktion  
$r_{XX}(\kappa)$  
auf den Bereich $|\kappa| \le 4$ beschränkt.

Dies ergibt die Werte:

- $r_{XX}(0) = 70$  
- $r_{XX}(\pm 1) = -56$  
- $r_{XX}(\pm 2) = 28$  
- $r_{XX}(\pm 3) = -8$  
- $r_{XX}(\pm 4) = 1$  
- $r_{XX}(\kappa) = 0$ für $|\kappa| \ge 5$

Beachten Sie, dass $H(z)$ eine vierfache Nullstelle bei $z = 1$ besitzt,  
was einem Hochpassverhalten entspricht.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_c1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_c2.png" style="width:45%;">
</div>

---

### d)

Mit dem in Python implementierten Burg-Algorithmus  
wird das MA-Modell $H(z)$  
durch AR-Modelle $1/A_n(z)$  
unterschiedlicher Ordnung $n = 2, 6, 32$ approximiert.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d12.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d22.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d32.png" style="width:45%;">
</div>

Die Ergebnisse zeigen, dass der Burg-Algorithmus die Nullstellen  
durch eine Konstellation von Polen nahe dem Einheitskreis approximiert.

Bei niedriger Modellordnung ($n = 2$) werden die Pole  
möglichst weit von der Nullstelle entfernt platziert  
und auf einem Kreis in Richtung der Nullstelle angeordnet,  
wobei der unmittelbare Bereich um die Nullstelle frei bleibt.

Anhand der geschätzten Leistungsdichtespektren ist erkennbar,  
dass bei Verwendung höherer AR-Modellordnungen  
eine zufriedenstellende Approximation erreicht wird.